# Matched-token QM9 LM pretraining ablation


Select **Runtime → Change runtime type → A100 GPU**. Run cells in order.
All artifacts live under `/content/drive/MyDrive/YLM_runs/<RUN_NAME>`.
Fixed names intentionally resume; change the name for a new experiment. Completed
`metrics.json` arms are skipped. Do not change budgets/model/seeds while resuming.

`src/profiling.py` records throughput, phase totals and per-step times, peak CUDA
memory, GPU utilization, a bounded `torch.profiler` trace, and `summary.json`.
GPU telemetry can be unavailable; profiles cover the script's profiled region,
not setup/download time. Reruns overwrite the individual invocation's profile.

**Time estimates (not measured on Colab):** allow 10–45 minutes for setup and 20k CPU xTB labels, and roughly 1–8 hours for six A100 arms at the default 1M tokens each. Geometry arms, HF download speed, numeric generation and shared CPU can make this longer. Profile the first arm before scaling. `small` trains from scratch; the default initializes Qwen weights. Budget is the script’s token accounting, including geometry prefixes; inspect `tokens_seen` and `budget_shortfall` rather than assuming identical steps.

In [ ]:
RUN_NAME = 'ylm_lm_pretrain_ablation' #@param {type:"string"}
MODEL = 'hf:Qwen/Qwen2.5-0.5B' #@param ['hf:Qwen/Qwen2.5-0.5B', 'small']
SEEDS = [0]
ARMS = ['smiles', 'smiles_xyz', 'prefixA', 'xattnA', 'xattnB', 'xattnC']
TOKENS_BUDGET = 1_000_000  # identical across every arm/seed
BATCH_SIZE = 2
MAX_LEN = 512
ENCODER = 'tiny'
CKPT_EVERY = 50
INSTALL_MP_API = False #@param {type:"boolean"}
SETTINGS = dict(model=MODEL, seeds=SEEDS, arms=ARMS, tokens_budget=TOKENS_BUDGET,
                batch_size=BATCH_SIZE, max_len=MAX_LEN, encoder=ENCODER, lr=3e-4)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, urllib.request, zipfile

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/YLM_runs')
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME
SAVE = BASE / RUN_NAME
SAVE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/YLM')
BRANCH = 'feat/llm-irreps-ablation-oct30'
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/yaniguan/YLM.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
packages = ['e3nn==0.6.0', 'numpy<2.3', 'scipy', 'rdkit', 'tblite',
            'transformers>=4.40,<5', 'accelerate', 'scikit-learn', 'pynvml', 'pytest']
if INSTALL_MP_API:
    packages.append('mp-api')
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)
import torch
torch.set_num_threads(2)
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(), '| output:', SAVE)
if 'A100' not in torch.cuda.get_device_name():
    print('Warning: estimates assume A100; this runtime has a different GPU.')

def run_script(script, *args):
    command = [sys.executable, str(ROOT / 'scripts' / script), *map(str, args)]
    print(' '.join(command), flush=True)
    with (SAVE / 'commands.log').open('a') as log:
        log.write(' '.join(command) + '\n')
    subprocess.run(command, cwd=ROOT, check=True)

# Preserve experiment settings on reruns; use a new RUN_NAME for changed settings.
settings = dict(SETTINGS, branch=BRANCH)
manifest = SAVE / 'notebook_config.json'
if manifest.exists():
    assert json.loads(manifest.read_text()) == settings, 'Settings changed: choose a new RUN_NAME.'
else:
    manifest.write_text(json.dumps(settings, indent=2))
(SAVE / 'git_commit.txt').write_text(subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True))

`--resume` starts fresh when no saved state exists and resumes existing checkpoints with saved configuration validation.

In [ ]:
LABEL_RUN = SAVE

# Keep the absolute local data paths stable across Colab sessions: saved configs
# use /content/YLM/data/... and posttraining reads those same paths.
data = ROOT / 'data'
data.mkdir(exist_ok=True)
if not (data / 'gdb9.sdf').exists():
    archive = data / 'qm9.zip'
    if not archive.exists():
        temporary = archive.with_suffix('.download')
        urllib.request.urlretrieve(
            'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/molnet_publish/qm9.zip', temporary)
        temporary.replace(archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(data)
assert (data / 'gdb9.sdf').exists()
label_cache = LABEL_RUN / 'qm9_xtb.pkl'
if label_cache.exists():
    shutil.copy2(label_cache, data / 'qm9_xtb.pkl')
else:
    run_script('01_make_labels.py', '--n', 20000, '--workers', os.cpu_count() or 2,
               '--seed', 0, '--sdf', data / 'gdb9.sdf', '--out', data / 'qm9_xtb.pkl')
    label_cache.parent.mkdir(parents=True, exist_ok=True)
    temporary = label_cache.with_suffix('.tmp')
    shutil.copy2(data / 'qm9_xtb.pkl', temporary)
    temporary.replace(label_cache)
if label_cache != SAVE / 'qm9_xtb.pkl' and not (SAVE / 'qm9_xtb.pkl').exists():
    shutil.copy2(label_cache, SAVE / 'qm9_xtb.pkl')

In [ ]:
for seed in SEEDS:
    for arm in ARMS:
        name = f'{arm}_seed{seed}'
        out = SAVE / 'pretrain' / name
        if (out / 'metrics.json').exists():
            print('Skip completed:', name)
            continue
        run_script('20_pretrain_lm.py', '--arm', arm, '--model', MODEL, '--encoder', ENCODER,
                   '--seed', seed, '--tokens_budget', TOKENS_BUDGET, '--batch_size', BATCH_SIZE,
                   '--max_len', MAX_LEN, '--lr', 3e-4, '--ckpt_every', CKPT_EVERY,
                   '--sdf', ROOT / 'data/gdb9.sdf', '--qm9', ROOT / 'data/qm9_xtb.pkl',
                   '--device', 'cuda', '--out', out, '--resume',
                   '--profile_dir', SAVE / 'profiling' / name)

In [ ]:
# Final cell: retain individual summaries and aggregate without including itself.
profile_root = SAVE / 'profiling'
profile_root.mkdir(exist_ok=True)
summaries = {}
for path in sorted(profile_root.rglob('summary.json')):
    if path != profile_root / 'summary.json':
        summaries[str(path.parent.relative_to(profile_root))] = json.loads(path.read_text())
combined = {'runs': summaries, 'run_count': len(summaries),
            'note': 'Individual summaries describe the latest invocation, not cumulative resumed sessions.'}
temporary = profile_root / 'summary.tmp'
temporary.write_text(json.dumps(combined, indent=2))
temporary.replace(profile_root / 'summary.json')
print('Profiling:', profile_root / 'summary.json')